# Adaptive AM-FM Decomposition of Speech for Parkinson's Disease Classification
## XGBoost classification on the PC-GITA vowels

Classifies speakers with Parkinson's disease (PD) versus healthy controls (HC) from sustained vowels, using eaQHM harmonic AM/FM variation features ($H_1$–$H_5$) together with the normalised first differences of $f_0$ and $A_1$ and spectral / Teager-energy descriptors (18 features).

An XGBoost classifier is evaluated with the same repeated nested speaker-independent cross-validation as the SVM notebook: predefined folds (`master_cv_folds_vowels.csv`, 5 repeats × 10 outer folds, stratified by label and gender), with a 5-fold inner grid search over the number of trees, learning rate, maximum depth, row subsampling and column subsampling (ROC AUC). Results are reported at sample and speaker level.

### 0. Data preparation — load the eaQHM features

In [1]:
import pandas as pd
from scipy.io import loadmat

# ==========================================
# 0. DATA PREP
# ==========================================
task = "vowels"
data_aeiou = loadmat(f'../features/pc_gita_{task}_16k_5ms_chopped_650.mat')
data_aeiou = data_aeiou['results'].squeeze()
data_aeiou = pd.DataFrame(data_aeiou)
data_aeiou.columns = ['centroid_mean', 'centroid_std', 'spectral_flux_mean', 'spectral_flux_max',
                       'teo_mean', 'teo_std', 'am_fm_corr', 'ampl_var', 'freq_var', 'f0_var',
                       'SRER', 'f0_norm_diff', 'jitter_T', 'A1_norm_diff', 'spectral_slope',
                       'f0_entropy', 'name']

file_names = data_aeiou['name'].apply(lambda x: x[0])
data = data_aeiou.copy()

### 0a. Unpack scalar features

In [2]:
data['centroid_mean']      = data['centroid_mean'].apply(lambda x: x[0][0])
data['centroid_std']       = data['centroid_std'].apply(lambda x: x[0][0])
data['spectral_flux_mean'] = data['spectral_flux_mean'].apply(lambda x: x[0][0])
data['spectral_flux_max']  = data['spectral_flux_max'].apply(lambda x: x[0][0])
data['teo_mean']           = data['teo_mean'].apply(lambda x: x[0][0])
data['teo_std']            = data['teo_std'].apply(lambda x: x[0][0])
data['am_fm_corr']         = data['am_fm_corr'].apply(lambda x: x[0][0])
data['f0_var']             = data['f0_var'].apply(lambda x: x[0][0])
data['SRER']               = data['SRER'].apply(lambda x: x[0][0])
data['f0_norm_diff']       = data['f0_norm_diff'].apply(lambda x: x[0][0])
data['jitter_T']           = data['jitter_T'].apply(lambda x: x[0][0])
data['A1_norm_diff']       = data['A1_norm_diff'].apply(lambda x: x[0][0])
data['spectral_slope']     = data['spectral_slope'].apply(lambda x: x[0][0])
data['f0_entropy']         = data['f0_entropy'].apply(lambda x: x[0][0])

### 0b. Speaker IDs, labels and gender

In [3]:
data['speaker'] = file_names.str.extract(r'(AVPEPUDEA[C]?\d{4})')[0]

gender_data = pd.read_csv("../gender_metadata/genders_pc_gita.csv", sep=",", header=0)
data = data.merge(gender_data, how="left", on="speaker")
data['label']  = data['speaker'].str.contains('C').map({True: 0, False: 1})
data['gender'] = data['gender'].map({'female': 0, 'male': 1})

### 0c. Harmonic feature expansion and feature selection

In [4]:
ampl_expanded = data['ampl_var'].apply(lambda x: x.flatten())
freq_expanded = data['freq_var'].apply(lambda x: x.flatten())
for i in range(5):
    data[f'ampl_var_H{i+1}'] = ampl_expanded.apply(lambda v: v[i])
    data[f'freq_var_H{i+1}'] = freq_expanded.apply(lambda v: v[i])

data = data.drop(columns=['jitter_T', 'ampl_var', 'freq_var', 'am_fm_corr',
                           'f0_entropy', 'f0_var', 'SRER', 'spectral_slope'])
data['vowel'] = file_names.str.extract(r'AVPEPUDEA[C]?\d{4}([aeiouAEIOU])')[0].str.upper()

### Imports for modelling

In [6]:
import pandas as pd
import numpy as np
import warnings
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedGroupKFold, GridSearchCV
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                             confusion_matrix, precision_score, recall_score,
                             brier_score_loss, log_loss)

warnings.filterwarnings("ignore", category=UserWarning)

### Helper function

In [7]:
# ==========================================
# HELPER
# ==========================================
def aggregate_mean_by_group(y, p, g):
    y, p, g = np.asarray(y).astype(int), np.asarray(p).astype(float), np.asarray(g)
    uniq = np.unique(g)
    y_g = np.zeros(len(uniq), dtype=int)
    p_g = np.zeros(len(uniq), dtype=float)
    for i, gg in enumerate(uniq):
        idx = np.where(g == gg)[0]
        p_g[i] = float(np.mean(p[idx])) if len(idx) else float('nan')
        y_g[i] = int(np.mean(y[idx]) >= 0.5) if len(idx) else 0
    return y_g, p_g, uniq

### 1. Load folds and merge

In [8]:
# ==========================================
# 1. LOAD FOLDS AND MERGE INTO DATA
# ==========================================
data['speaker'] = file_names.str.extract(r'(AVPEPUDEA[C]?\d{4})')[0]
data['speaker'] = data['speaker'].astype(str).str.strip().str.upper()

fold_file = f"../folds/master_cv_folds_{task}.csv"
print(f"Loading predefined folds from {fold_file}...")
fold_map = pd.read_csv(fold_file)
fold_map['speaker'] = fold_map['speaker'].astype(str).str.strip().str.upper()

repeat_fold_cols = [c for c in fold_map.columns if c.startswith("Repeat_") and c.endswith("_Fold")]
speaker_folds = fold_map[['speaker'] + repeat_fold_cols].drop_duplicates(subset='speaker')

data = data.merge(speaker_folds, on='speaker', how='inner').reset_index(drop=True)

if data.empty:
    raise ValueError("CRITICAL: DataFrame empty after fold merge — check speaker ID format")

print(f"Data shape after fold merge: {data.shape}")
print(f"Fold columns detected: {repeat_fold_cols}")

for col in repeat_fold_cols:
    assert data.groupby('speaker')[col].nunique().max() == 1, \
        f"Speaker has inconsistent fold assignments in {col}"
print("✅ All speakers have consistent fold assignments")

original_speakers = set(fold_map['speaker'].str.strip().str.upper())
new_speakers = set(data['speaker'].str.strip().str.upper())
missing = new_speakers - original_speakers
if missing:
    print(f"⚠️  {len(missing)} speakers in vowels not in fold file: {missing}")

Loading predefined folds from ../folds/master_cv_folds_vowels.csv...
Data shape after fold merge: (1500, 28)
Fold columns detected: ['Repeat_1_Fold', 'Repeat_2_Fold', 'Repeat_3_Fold', 'Repeat_4_Fold', 'Repeat_5_Fold']
✅ All speakers have consistent fold assignments


### 2. Features and groups

In [9]:
# ==========================================
# 2. FINALIZE FEATURES AND GROUPS
# ==========================================
groups = data['speaker'].values

data['stratify_key'] = data['label'].astype(str) + "_" + data['gender'].astype(str)

cols_to_drop = ['label', 'gender', 'stratify_key', 'speaker', 'name',
                'vowel', 'sample_id'] + repeat_fold_cols
X = data.drop(columns=cols_to_drop, errors='ignore')
y = data['label']
y_stratify = data['stratify_key']

feature_cols = list(X.columns)

N_REPEATS      = len(repeat_fold_cols)
N_OUTER_SPLITS = data[repeat_fold_cols[0]].nunique()
N_INNER_SPLITS = 5
base_random_state = 42

print(f"\nFeature matrix : {X.shape}")
print(f"Features       : {feature_cols}")
print(f"Unique speakers: {len(np.unique(groups))}")
print(f"Repeats x Folds: {N_REPEATS} x {N_OUTER_SPLITS}")
print("Stratify key counts:"); print(y_stratify.value_counts().sort_index())


Feature matrix : (1500, 18)
Features       : ['centroid_mean', 'centroid_std', 'spectral_flux_mean', 'spectral_flux_max', 'teo_mean', 'teo_std', 'f0_norm_diff', 'A1_norm_diff', 'ampl_var_H1', 'freq_var_H1', 'ampl_var_H2', 'freq_var_H2', 'ampl_var_H3', 'freq_var_H3', 'ampl_var_H4', 'freq_var_H4', 'ampl_var_H5', 'freq_var_H5']
Unique speakers: 100
Repeats x Folds: 5 x 10
Stratify key counts:
stratify_key
0_0    375
0_1    375
1_0    375
1_1    375
Name: count, dtype: int64


### 3. Result storage

In [10]:
# ==========================================
# 3. RESULT STORAGE
# ==========================================
metrics_keys = ["accuracy", "f1", "auc", "precision", "recall", "brier", "log_loss"]
results_sample  = {k: [] for k in metrics_keys}
results_speaker = {k: [] for k in metrics_keys}
conf_matrices_sample  = []
conf_matrices_speaker = []

# Per-fold (fold-to-fold) records, one row per (repeat, outer fold)
fold_records = []

### 4. Repeated nested cross-validation

In [ ]:
# ==========================================
# 4. MAIN LOOP
# ==========================================
print(f"\nStarting XGBoost Repeated Nested CV ({N_REPEATS} Repeats × {N_OUTER_SPLITS} Folds)…")

for repeat_idx, repeat_col in enumerate(repeat_fold_cols):
    repeat       = repeat_idx + 1
    current_seed = base_random_state + repeat_idx

    print(f"\n{'='*80}")
    print(f"REPEAT {repeat}/{N_REPEATS}  (Seed: {current_seed} | Column: {repeat_col})")
    print("="*80)

    for fold_id in range(N_OUTER_SPLITS):

        test_mask  = (data[repeat_col] == fold_id).values
        train_mask = ~test_mask

        X_train, X_test = X.iloc[train_mask], X.iloc[test_mask]
        y_train, y_test = y.iloc[train_mask], y.iloc[test_mask]
        g_train_np      = groups[train_mask]
        g_test_np       = groups[test_mask]
        y_strat_train   = y_stratify.iloc[train_mask]

        # LEAKAGE CHECK 1: Outer speaker overlap 
        outer_overlap = set(g_train_np).intersection(set(g_test_np))
        assert len(outer_overlap) == 0, \
            f"❌ OUTER LEAKAGE R{repeat} Fold {fold_id}: {outer_overlap}"

        # LEAKAGE CHECK 2: Speaker counts
        n_spk_train = len(set(g_train_np))
        n_spk_test  = len(set(g_test_np))
        assert n_spk_train + n_spk_test == len(np.unique(groups)), \
            f"❌ Speaker count mismatch: {n_spk_train}+{n_spk_test} != {len(np.unique(groups))}"

        # LEAKAGE CHECK 3: Sample index overlap 
        train_indices = np.where(train_mask)[0]
        test_indices  = np.where(test_mask)[0]
        assert len(set(train_indices) & set(test_indices)) == 0, \
            f"❌ SAMPLE OVERLAP R{repeat} Fold {fold_id}"

        # LEAKAGE CHECK 4: Forbidden columns in X 
        forbidden   = ["label", "speaker", "speaker_id", "gender", "stratify_key", "name"]
        leaked_cols = [c for c in forbidden if c in X_train.columns]
        assert len(leaked_cols) == 0, \
            f"❌ Forbidden columns in features: {leaked_cols}"

        # LEAKAGE CHECK 5: Inner CV speaker overlap 
        inner_cv_check = StratifiedGroupKFold(
            n_splits=N_INNER_SPLITS, shuffle=True,
            random_state=current_seed + fold_id,
        ).split(X_train, y_strat_train, g_train_np)

        for i, (tr_i, va_i) in enumerate(inner_cv_check):
            overlap_inner = set(g_train_np[tr_i]) & set(g_train_np[va_i])
            assert len(overlap_inner) == 0, \
                f"❌ INNER LEAKAGE R{repeat} Fold {fold_id} Inner {i}: {overlap_inner}"

        print(f"R{repeat} Fold {fold_id+1:2d}: ✅ All checks passed | "
              f"Train: {len(X_train)} samples ({n_spk_train} spk) | "
              f"Test: {len(X_test)} samples ({n_spk_test} spk)")

        # GridSearchCV 
        xgb = XGBClassifier(
            objective='binary:logistic',
            eval_metric='auc',
            random_state=current_seed,
            n_jobs=1,
        )

        param_grid = {
            "n_estimators":     [100, 200, 300, 500],
            "learning_rate":    [0.01, 0.05, 0.1],
            "max_depth":        [3, 5, 7],
            "subsample":        [0.5, 0.8, 1.0],
            "colsample_bytree": [0.5, 0.8, 1.0],
        }

        inner_cv = StratifiedGroupKFold(
            n_splits=N_INNER_SPLITS, shuffle=True,
            random_state=current_seed + fold_id,
        ).split(X_train, y_strat_train, g_train_np)

        grid_search = GridSearchCV(
            estimator=xgb,
            param_grid=param_grid,
            cv=inner_cv,
            scoring="roc_auc",
            n_jobs=-1,
            verbose=0,
        )
        grid_search.fit(X_train, y_train)
        best_model  = grid_search.best_estimator_
        bp          = best_model.get_params()

        # Sample-level predictions 
        y_pred_sample  = best_model.predict(X_test)
        y_proba_sample = best_model.predict_proba(X_test)[:, 1]

        results_sample["accuracy"].append(  accuracy_score(  y_test, y_pred_sample))
        results_sample["f1"].append(        f1_score(        y_test, y_pred_sample))
        results_sample["auc"].append(       roc_auc_score(   y_test, y_proba_sample))
        results_sample["precision"].append( precision_score( y_test, y_pred_sample))
        results_sample["recall"].append(    recall_score(    y_test, y_pred_sample))
        results_sample["brier"].append(     brier_score_loss(y_test, y_proba_sample))
        results_sample["log_loss"].append(  log_loss(        y_test, y_proba_sample))
        conf_matrices_sample.append(confusion_matrix(y_test, y_pred_sample))

        # Speaker-level predictions 
        y_test_spk, y_proba_spk, _ = aggregate_mean_by_group(
            y_test.values, y_proba_sample, g_test_np
        )
        y_pred_spk = (y_proba_spk >= 0.5).astype(int)

        results_speaker["accuracy"].append(  accuracy_score(  y_test_spk, y_pred_spk))
        results_speaker["f1"].append(        f1_score(        y_test_spk, y_pred_spk))
        results_speaker["auc"].append(       roc_auc_score(   y_test_spk, y_proba_spk))
        results_speaker["precision"].append( precision_score( y_test_spk, y_pred_spk))
        results_speaker["recall"].append(    recall_score(    y_test_spk, y_pred_spk))
        results_speaker["brier"].append(     brier_score_loss(y_test_spk, y_proba_spk))
        results_speaker["log_loss"].append(  log_loss(        y_test_spk, y_proba_spk))
        conf_matrices_speaker.append(confusion_matrix(y_test_spk, y_pred_spk))

        fold_records.append({
            "repeat":               repeat,
            "fold":                 fold_id + 1,
            "seed":                 current_seed,
            "best_n_estimators":    bp['n_estimators'],
            "best_learning_rate":   bp['learning_rate'],
            "best_max_depth":       bp['max_depth'],
            "best_subsample":       bp['subsample'],
            "best_colsample_bytree": bp['colsample_bytree'],
            "n_train_samples":      len(X_train),
            "n_test_samples":       len(X_test),
            "n_train_speakers":     n_spk_train,
            "n_test_speakers":      n_spk_test,
            "sample_accuracy":      results_sample["accuracy"][-1],
            "sample_f1":            results_sample["f1"][-1],
            "sample_auc":           results_sample["auc"][-1],
            "sample_precision":     results_sample["precision"][-1],
            "sample_recall":        results_sample["recall"][-1],
            "sample_log_loss":      results_sample["log_loss"][-1],
            "sample_brier":         results_sample["brier"][-1],
            "speaker_accuracy":     results_speaker["accuracy"][-1],
            "speaker_f1":           results_speaker["f1"][-1],
            "speaker_auc":          results_speaker["auc"][-1],
            "speaker_precision":    results_speaker["precision"][-1],
            "speaker_recall":       results_speaker["recall"][-1],
            "speaker_log_loss":     results_speaker["log_loss"][-1],
            "speaker_brier":        results_speaker["brier"][-1],
        })

        print(f"           Est={bp['n_estimators']}, LR={bp['learning_rate']}, "
              f"Depth={bp['max_depth']}, Sub={bp['subsample']}, ColSample={bp['colsample_bytree']}")
        print(f"           [Sample]  Acc: {results_sample['accuracy'][-1]:.4f} | "
              f"F1: {results_sample['f1'][-1]:.4f} | AUC: {results_sample['auc'][-1]:.4f} | "
              f"Brier: {results_sample['brier'][-1]:.4f}")
        print(f"           [Speaker] Acc: {results_speaker['accuracy'][-1]:.4f} | "
              f"F1: {results_speaker['f1'][-1]:.4f} | AUC: {results_speaker['auc'][-1]:.4f} | "
              f"Brier: {results_speaker['brier'][-1]:.4f}")
        print("-" * 60)

### 5. Save per-fold results

In [ ]:
# ==========================================
# 5. SAVE FOLD-TO-FOLD RESULTS
# ==========================================
fold_df = pd.DataFrame(fold_records)

fold_csv = f"fold_results_xgb_{task}.csv"
fold_df.to_csv(fold_csv, index=False)
print(f"\nSaved per-fold results → {fold_csv}")

fold_txt = f"fold_results_xgb_{task}.txt"
with open(fold_txt, "w", encoding="utf-8") as f:
    f.write(f"XGBoost — ALL FEATURES ({len(feature_cols)} features) — PC-GITA {task}\n")
    f.write(f"Features: {feature_cols}\n")
    f.write(f"Repeats x Folds: {N_REPEATS} x {N_OUTER_SPLITS}\n")
    f.write("=" * 100 + "\n")
    f.write(fold_df.to_string(index=False))
    f.write("\n")
print(f"Saved per-fold results → {fold_txt}")

### 6. Summary and confusion matrices

In [ ]:
# ==========================================
# 6. SUMMARY
# ==========================================
total_folds = N_REPEATS * N_OUTER_SPLITS
print(f"\n{'='*55}")
print(f"FINAL XGBOOST RESULTS ({total_folds} Total Folds)")
print("="*55)
print(f"{'Metric':<12} | {'Sample-Level':<25} | {'Speaker-Level':<25}")
print("-" * 68)

for metric in metrics_keys:
    ms = np.mean(results_sample[metric]);  ss = np.std(results_sample[metric])
    mk = np.mean(results_speaker[metric]); sk = np.std(results_speaker[metric])
    print(f"{metric.capitalize():<12} | {ms:.4f} ± {ss:.4f}            | {mk:.4f} ± {sk:.4f}")

summary_rows = []
for level, r in (("sample", results_sample), ("speaker", results_speaker)):
    row = {"level": level}
    for metric in metrics_keys:
        row[f"{metric}_mean"] = np.mean(r[metric])
        row[f"{metric}_std"]  = np.std(r[metric])
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
summary_csv = f"results_xgb_{task}.csv"
summary_df.to_csv(summary_csv, index=False)
print(f"\nSaved aggregated summary → {summary_csv}")

print("\n--- Aggregated Confusion Matrix (Sample-Level) ---")
print(np.sum(conf_matrices_sample, axis=0))
print("\n--- Aggregated Confusion Matrix (Speaker-Level) ---")
print(np.sum(conf_matrices_speaker, axis=0))